# Обучение всех 5 экспериментов в одном сеансе

Один ноутбук — установка один раз, потом цикл по 5 генераторам синтетики.

**Параметры**: Qwen2.5-Coder-7B-Instruct (4bit), r=16, alpha=32, lr=2e-4, 1 epoch,
batch=1×8, **max_seq_length=12288** (вмещает весь промпт ~10k токенов + ответ).

**Данные**: `MyDrive/text2sql/<exp>/card_games/{train,val}.jsonl`
**Адаптеры**: сохраняются в `MyDrive/text2sql/<exp>/adapter/`

**Запуск**: ячейка 0 (install) → Среда выполнения → Перезапустить сеанс → остальные по порядку.

GPU: A100 40GB. Время: ~30-45 мин на модель × 5 ≈ 3 часа.

In [ ]:
# 0. Установка (после неё — Среда выполнения → Перезапустить сеанс)
!pip install -q --upgrade pip
!pip install -q --upgrade unsloth unsloth_zoo
print('✓ install done — ПЕРЕЗАПУСТИ СЕАНС, потом запускай ячейку 1')

## 1. Импорты + Drive (после перезапуска сеанса)

In [ ]:
from unsloth import FastLanguageModel   # ОБЯЗАТЕЛЬНО первым
import torch, gc, json, os
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/text2sql'
MAX_SEQ_LEN = 12288
BASE_MODEL = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit'

# ⏱ Полный датасет (1.0). Для дедлайн-демо использовали 0.25 — теперь финальный прогон.
# На полных ~800 примерах одна модель тренируется ~40 мин на A100.
SAMPLE_FRAC = 1.0

# resume: если адаптер уже есть — пропускаем эксперимент.
# Снимай галочку, чтобы пересчитать конкретный.
SKIP_IF_EXISTS = True

# Имена экспериментов — из реестра (единый источник правды).
reg = json.load(open(f'{BASE}/registry.json'))
EXPERIMENTS = [e['name'] for e in reg['experiments']]
print('✓ torch:', torch.__version__)
print(f'✓ SAMPLE_FRAC = {SAMPLE_FRAC}  (1.0 = весь датасет)')
print(f'✓ SKIP_IF_EXISTS = {SKIP_IF_EXISTS}')
print(f'✓ experiments: {EXPERIMENTS}')

## 2. Функция обучения одного эксперимента

In [ ]:
def train_one(exp):
    adapter_dir = f'{BASE}/{exp}/adapter'

    # resume: если адаптер уже сохранён — пропускаем
    if SKIP_IF_EXISTS and os.path.exists(f'{adapter_dir}/adapter_config.json'):
        print(f'⏭  {exp}: адаптер уже есть в {adapter_dir} — пропускаю')
        return

    print(f'\n{"="*60}\nОБУЧЕНИЕ: {exp}\n{"="*60}')
    data_dir = f'{BASE}/{exp}'
    ckpt_dir = f'{BASE}/{exp}/checkpoints'

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN,
        dtype=None, load_in_4bit=True)

    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32,
        target_modules=['q_proj','k_proj','v_proj','o_proj',
                        'gate_proj','up_proj','down_proj'],
        lora_dropout=0.05, bias='none',
        use_gradient_checkpointing='unsloth', random_state=42)

    # eval выключен (OOM-фикс на max_seq=12288)
    raw = load_dataset('json', data_files={'train': f'{data_dir}/train.jsonl'})

    # сабсэмпл (для дедлайна был 0.25, теперь 1.0 — весь датасет)
    if SAMPLE_FRAC < 1.0:
        n_full = len(raw['train'])
        n = max(1, int(n_full * SAMPLE_FRAC))
        raw['train'] = raw['train'].shuffle(seed=42).select(range(n))
        print(f'  субсэмпл: {n}/{n_full} ({SAMPLE_FRAC*100:.0f}%)')

    def fmt(ex):
        return {'text': tokenizer.apply_chat_template(
            ex['messages'], tokenize=False, add_generation_prompt=False)}
    ds = raw.map(fmt, remove_columns=['messages','_meta'])
    print(f'  train={len(ds["train"])}')

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=ds['train'],
        args=SFTConfig(
            output_dir=ckpt_dir,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            num_train_epochs=1, learning_rate=2e-4,
            warmup_ratio=0.05, bf16=True,
            logging_steps=10,
            save_strategy='steps', save_steps=50, save_total_limit=2,
            eval_strategy='no',
            max_seq_length=MAX_SEQ_LEN, dataset_text_field='text',
            packing=False, report_to='none', seed=42,
        ),
    )
    trainer.train()

    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f'✓ {exp} — адаптер сохранён в {adapter_dir}')

    del model, tokenizer, trainer, ds, raw
    torch.cuda.empty_cache(); gc.collect()

## 3. По одной ячейке на эксперимент

Запускай по порядку. Можешь остановиться после любой и идти на этап инференса — адаптер уже сохранён.

**Порядок** (gpt41_bi первым — для дедлайна):
1. gpt41_bi ← запусти и иди делать eval, остальные пусть докрутятся фоном
2. codexmini_bi
3. gpt41nano_bi
4. gpt4omini_bi
5. gpt41mini_bi

In [ ]:
# 1️⃣ gpt41_bi — приоритетный, запускай сразу. ~10 мин с SAMPLE_FRAC=0.25
try:
    train_one('gpt41_bi')
except Exception as e:
    print(f'✗ gpt41_bi УПАЛ: {e}')
    torch.cuda.empty_cache(); gc.collect()

In [ ]:
# 2️⃣ codexmini_bi
try:
    train_one('codexmini_bi')
except Exception as e:
    print(f'✗ codexmini_bi УПАЛ: {e}')
    torch.cuda.empty_cache(); gc.collect()

In [ ]:
# 3️⃣ gpt41nano_bi
try:
    train_one('gpt41nano_bi')
except Exception as e:
    print(f'✗ gpt41nano_bi УПАЛ: {e}')
    torch.cuda.empty_cache(); gc.collect()

In [ ]:
# 4️⃣ gpt4omini_bi
try:
    train_one('gpt4omini_bi')
except Exception as e:
    print(f'✗ gpt4omini_bi УПАЛ: {e}')
    torch.cuda.empty_cache(); gc.collect()

In [ ]:
# 5️⃣ gpt41mini_bi
try:
    train_one('gpt41mini_bi')
except Exception as e:
    print(f'✗ gpt41mini_bi УПАЛ: {e}')
    torch.cuda.empty_cache(); gc.collect()

print('\n✓✓✓ Все эксперименты завершены')